<a href="https://colab.research.google.com/github/Jericho301/SmallscaleWeather/blob/main/bob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import requests
from openai import OpenAI
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime, timezone # Added timezone import

# API defining
API_KEY = userdata.get("OPEN_ROUTER_KEY")
API_Weather = userdata.get("bobby")

client = OpenAI(api_key=API_KEY, base_url="https://openrouter.ai/api/v1/")
State = input("Input name of any City to obtain its current temperature:")

def get_weather(city_name):
    geo_url = f"http://api.openweathermap.org/geo/1.0/direct?q={city_name}&limit=1&appid={API_Weather}"
    geo_response = requests.get(geo_url)
    geo_data = geo_response.json()

    if not geo_data:
        return {
            "error": f"City '{city_name}' not found. Try format like 'Washington, DC'"
        }

    lat = geo_data[0]["lat"]
    lon = geo_data[0]["lon"]

    weather_url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_Weather}&units=metric"
    weather_response = requests.get(weather_url)
    weather_data = weather_response.json()

    city = weather_data.get("name")
    temperature = weather_data.get("main", {}).get("temp")
    description = weather_data.get("weather", [{}])[0].get("description")

    timeshit = weather_data.get("dt")
    timeshit_diff = weather_data.get("timezone", 0)
    local_time = datetime.fromtimestamp(timeshit + timeshit_diff, tz=timezone.utc) # Modified to use timezone.utc
    form_local_time = local_time.strftime("%Y-%m-%d %H:%M:%S")

    return {
        "city": city,
        "temperature": f"{temperature}ºC",
        "description": description,
        "local_time": form_local_time
    }

#tools definition to accept city name
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Retrieves current weather for the given city name.",
        "parameters": {
            "type": "object",
            "properties": {
                "city_name": {
                    "type": "string",
                    "description": "City name, e.g., 'Paris' or 'London, UK'"
                }
            },
            "required": ["city_name"],
            "additionalProperties": False
        },
        "strict": True
    }
}]

# User can now just ask for a city
messages = [
    {"role": "system", "content": "You are a helpful weather assistant"},
    {"role": "user", "content": f"What is the weather in {State}"}
]

#AI calling function with city name
completion = client.chat.completions.create(
    model = "openai/gpt-3.5-turbo",
    messages = messages,
    tools = tools,
)

def call_function(name, args):
    if name == "get_weather":
        return get_weather(args["city_name"])

# Process the function call
if hasattr(completion.choices[0].message, 'tool_calls') and completion.choices[0].message.tool_calls:
    for tool_call in completion.choices[0].message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        messages.append(completion.choices[0].message)

        result = call_function(name, args)
        messages.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
else:
    print("AI didn't call function:", completion.choices[0].message.content)
    exit()

# Get final response
completion_two = client.chat.completions.create(
    model="openai/gpt-3.5-turbo",
    tools=tools,
    tool_choice="auto",
    messages=messages + [{"role": "user", "content": "Format the weather as JSON including city, temperature, description and local_time"}],
    response_format={"type": "json_object"}
)

print(completion_two.choices[0].message.content)


SecretNotFoundError: Secret OPEN_ROUTER_KEY does not exist.